# Community and vended tools

Assistant version 4. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_calculator_time_weather

Source: [01_calculator_time_weather.py](examples/01_calculator_time_weather.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""calculator, current_time, http_request and Open-Meteo; recorded offline weather.

Run: uv run python modules/06_community_tools/examples/01_calculator_time_weather.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No; live weather needs internet. Resources created: none.
"""

import asyncio

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message
from common.tools.travel_day import (
    WEATHER_URL,
    convert_budget,
    travel_clock,
    weather_client,
    weather_tool,
)


async def run(args, cfg):
    async with weather_client(args.mock) as client:
        http_tool = weather_tool(client)
        model = get_model(
            cfg,
            responses=[
                tool_message("http_request", {"method": "GET", "url": WEATHER_URL}),
                tool_message("convert_budget", {"amount_usd": "100"}, "money"),
                "Lisbon fixture: 18.5 C; EUR 92 at an assumed rate, not a live quote.",
            ],
        )
        agent = Agent(
            model=model,
            tools=[http_tool, convert_budget],
            callback_handler=None,
            system_prompt="Nimbus travel day. Current time: " + travel_clock(args.mock),
            retry_strategy=None,
        )
        # KEY IDEA: operator-controlled HTTP transport enforces the one allowed endpoint.
        return await agent.invoke_async(
            "Fetch " + WEATHER_URL + " and convert USD 100 to EUR "
            "at the assumed rate 0.92. Distinguish fixture weather from live weather.",
            limits={"turns": 4},
        )


def main() -> None:
    args, cfg = lesson_settings()
    console.print("Weather source:", "synthetic recorded shape" if args.mock else "Open-Meteo live")
    complete(asyncio.run(run(args, cfg)), cfg)


run_lesson(main, "modules/06_community_tools/examples/01_calculator_time_weather.py")

## 02_files_and_consent

Source: [02_files_and_consent.py](examples/02_files_and_consent.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""file_read, file_write, editor and shell with explicit consent and path restrictions.

Run: uv run python modules/06_community_tools/examples/02_files_and_consent.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import tempfile
from pathlib import Path
from unittest.mock import patch

from strands import Agent
from strands.vended_tools.file_editor import make_file_editor
from strands.vended_tools.shell import make_shell
from strands_tools import file_read, file_write

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.reviewed_workspace import ReviewedWorkspace


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-files-") as directory:
        root = Path(directory)
        consent = (lambda prompt: "yes") if args.mock else input
        backend = ReviewedWorkspace(root, consent)
        tools = [
            make_file_editor(sandbox=backend),
            make_shell(sandbox=backend),
            file_read,
            file_write,
        ]
        agent = Agent(
            model=get_model(cfg, responses=[] if args.mock else None),
            tools=tools,
            callback_handler=None,
        )
        # KEY IDEA: paths/operations are checked below the model, before consent and execution.
        console.print(
            agent.tool.file_editor(command="create", path="/notes.txt", file_text="Lisbon")
        )
        console.print(
            agent.tool.file_editor(
                command="str_replace",
                path="/notes.txt",
                old_str="Lisbon",
                new_str="Lisbon budget EUR 92",
            )
        )
        console.print(agent.tool.shell(command="nimbus-status"))
        console.print(agent.tool.file_read(path=str(root / "notes.txt"), mode="text"))
        # Demonstrate the community writer's own refusal path, never bypass its prompt.
        with patch.object(file_write, "get_user_input", return_value="declined for this demo"):
            result = agent.tool.file_write(path=str(root / "declined.txt"), content="Not written")
        assert not (root / "declined.txt").exists()
        console.print("Community file_write consent refusal:", result["status"])
    complete()


run_lesson(main, "modules/06_community_tools/examples/02_files_and_consent.py")

## 03_python_with_consent

Source: [03_python_with_consent.py](examples/03_python_with_consent.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Vended python_repl replacement, consent gate and BYPASS_TOOL_CONSENT explanation.

Run: uv run python modules/06_community_tools/examples/03_python_with_consent.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No; execution environment capability check. Resources created: none.
"""

import tempfile
from pathlib import Path

from strands import Agent
from strands.vended_tools.python_repl import make_python_repl

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.reviewed_workspace import APPROVED_CODE, ReviewedWorkspace


def main() -> None:
    args, cfg = lesson_settings()
    with tempfile.TemporaryDirectory(prefix="nimbus-python-") as directory:
        backend = ReviewedWorkspace(Path(directory), (lambda prompt: "no") if args.mock else input)
        repl = make_python_repl(sandbox=backend)
        agent = Agent(
            model=get_model(cfg, responses=[] if args.mock else None),
            tools=[repl],
            callback_handler=None,
        )
        # KEY IDEA: a supplied execution backend checks exact code and asks consent.
        denied = agent.tool.python_repl(code="import os; print(os.environ)")
        assert denied["status"] == "error"
        if args.mock:
            rejected = agent.tool.python_repl(code=APPROVED_CODE)
            assert rejected["status"] == "error"
            backend.consent = lambda prompt: "yes"
        result = agent.tool.python_repl(code=APPROVED_CODE)
        console.print(result)
        console.print(
            "The teaching backend accepts one fixed snippet; it is not arbitrary-code isolation."
        )
    complete()


run_lesson(main, "modules/06_community_tools/examples/03_python_with_consent.py")

## 04_tool_catalog

Source: [04_tool_catalog.py](examples/04_tool_catalog.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""retrieve, memory, think, use_llm, image_reader, journal, handoff_to_user; mix with custom tools.

Run: uv run python modules/06_community_tools/examples/04_tool_catalog.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Optional; AWS tool adapters use fixtures locally. Resources created: none.
"""

from contextlib import nullcontext
from unittest.mock import patch

from strands import Agent
from strands.vended_tools.notebook import notebook
from strands_tools import handoff_to_user, image_reader, use_agent

from common.data import data_path
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    console.print("Deprecated names and supported replacements: see MIGRATIONS.md in this module.")
    agent = Agent(
        model=get_model(cfg, responses=["Ask the traveler for dates before recommending."]),
        tools=[notebook, use_agent, image_reader, handoff_to_user],
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: session notes are private state until a tool returns their content.
    console.print(agent.tool.notebook(mode="create", name="trip", new_str="Lisbon; budget 600"))
    console.print(agent.tool.notebook(mode="read", name="trip"))
    console.print(
        agent.tool.image_reader(image_path=str(data_path("receipts/receipt_01.png")))["status"]
    )
    console.print(
        agent.tool.use_agent(
            prompt="What trip detail should we ask for next?",
            system_prompt="You are a concise Nimbus specialist.",
            tools=[],
        )
    )
    context = (
        patch.object(handoff_to_user, "get_user_input", return_value="Two travelers")
        if args.mock
        else nullcontext()
    )
    with context:
        console.print(agent.tool.handoff_to_user(message="How many travelers?"))
    complete()


run_lesson(main, "modules/06_community_tools/examples/04_tool_catalog.py")

## 05_read_only_aws

Source: [05_read_only_aws.py](examples/05_read_only_aws.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""use_aws with scoped read-only role; denied write demonstration.

Run: uv run python modules/06_community_tools/examples/05_read_only_aws.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Yes; recorded contract locally. Resources created: none.
"""

from contextlib import nullcontext
from unittest.mock import patch

import boto3
from botocore.stub import Stubber
from strands import Agent
from strands_tools import use_aws

from common.config import load_settings
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import console, run_cli


def guarded_call(agent, operation: str, region: str, profile: str) -> dict:
    # KEY IDEA: reject writes before delegating; IAM is a separate enforcement layer.
    if operation != "describe_regions":
        raise PermissionError("Only ec2.describe_regions is allowed")
    return agent.tool.use_aws(
        service_name="ec2",
        operation_name=operation,
        parameters={},
        region=region,
        profile_name=profile,
        label="Read region names",
    )


def main() -> None:
    args, cfg = lesson_settings()
    agent = Agent(
        model=get_model(cfg, responses=[] if args.mock else None),
        tools=[use_aws],
        callback_handler=None,
    )
    target = load_settings(provider="bedrock")
    if args.mock:
        client = boto3.client(
            "ec2",
            region_name=target.aws_region,
            aws_access_key_id="testing",
            aws_secret_access_key="testing",
        )
        stub = Stubber(client)
        stub.add_response("describe_regions", {"Regions": [{"RegionName": "us-east-1"}]}, {})
        context = patch.object(use_aws, "get_boto3_client", return_value=client)
    else:
        require_bedrock(cfg)
        context, stub = nullcontext(), nullcontext()
    with context, stub:
        console.print(
            guarded_call(agent, "describe_regions", target.aws_region, target.aws_profile)
        )
        try:
            guarded_call(agent, "terminate_instances", target.aws_region, target.aws_profile)
        except PermissionError as exc:
            console.print("Write denied:", str(exc))
    complete()


run_lesson(main, "modules/06_community_tools/examples/05_read_only_aws.py")

## travel_day_helper

Source: [travel_day_helper.py](use_case/travel_day_helper.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Prepare a Lisbon travel-day brief with constrained HTTP and a currency tool.

Run: uv run python modules/06_community_tools/use_case/travel_day_helper.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

import asyncio

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message
from common.tools.travel_day import (
    WEATHER_URL,
    convert_budget,
    travel_clock,
    weather_client,
    weather_tool,
)


async def run(args, cfg):
    async with weather_client(args.mock) as client:
        model = get_model(
            cfg,
            responses=[
                tool_message("http_request", {"method": "GET", "url": WEATHER_URL}),
                tool_message("convert_budget", {"amount_usd": "150"}, "convert"),
                "Lisbon weather fixture: 18.5 C, no precipitation. EUR 138 at the assumed rate.",
            ],
        )
        agent = Agent(
            model=model,
            tools=[weather_tool(client), convert_budget],
            system_prompt="Nimbus travel day; timestamp " + travel_clock(args.mock),
            callback_handler=None,
            retry_strategy=None,
        )
        # KEY IDEA: API requests and money calculations are explicit, separately checked tools.
        return await agent.invoke_async(
            "Use " + WEATHER_URL + " and convert USD 150 at 0.92 EUR/USD. "
            "Say that the exchange rate is assumed. Give a short departure-day brief.",
            limits={"turns": 4},
        )


def main() -> None:
    args, cfg = lesson_settings()
    console.print("Weather mode:", "synthetic fixture" if args.mock else "live Open-Meteo")
    complete(asyncio.run(run(args, cfg)), cfg)


run_lesson(main, "modules/06_community_tools/use_case/travel_day_helper.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
